CELL 1: IMPORT LIBRARIES

Explanation: We import all necessary libraries for data manipulation, preprocessing, modeling, evaluation, and serialization. Each library has a specific role in our pipeline.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Optional - install if you want XGBoost (bonus)
# pip install xgboost
# from xgboost import XGBRegressor

# Set display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries imported successfully!")

Libraries imported successfully!


What we conclude: All libraries are ready. If XGBoost import fails, we skip it (Random Forest is still fine).

CELL 2: LOAD PROCESSED DATASET

Explanation: We load the 12-feature dataset we created in Phase 2. This ensures consistency between EDA and modeling.

In [2]:
# Load the processed dataset
df = pd.read_csv('../data/processed/ames_selected.csv')

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nColumn names:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (2930, 13)

First 5 rows:
   Bedroom AbvGr  Gr Liv Area  Lot Area  Year Built  Garage Cars Neighborhood  \
0              3         1656     31770        1960         2.00        NAmes   
1              2          896     11622        1961         1.00        NAmes   
2              3         1329     14267        1958         1.00        NAmes   
3              3         2110     11160        1968         2.00        NAmes   
4              3         1629     13830        1997         2.00      Gilbert   

   Overall Cond  Overall Qual Bsmt Qual Heating Central Air  SalePrice  \
0             5             6        TA    GasA           Y     215000   
1             6             5        TA    GasA           Y     105000   
2             6             6        TA    GasA           Y     172000   
3             5             7        TA    GasA           Y     244000   
4             5             5        Gd    GasA           Y     189900   

   bat

What we conclude:

    Shape should be (2930, 13) - 12 features + SalePrice

    Columns match our selected features

CELL 3: SEPARATE FEATURES AND TARGET

Explanation: We separate the target variable (SalePrice) from the features. The target is what we want to predict. Features are the inputs.

In [3]:
# Separate features and target
X = df.drop('SalePrice', axis=1)
y = df['SalePrice']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures:\n{X.columns.tolist()}")
print(f"\nTarget statistics:")
print(y.describe())

Features shape: (2930, 12)
Target shape: (2930,)

Features:
['Bedroom AbvGr', 'Gr Liv Area', 'Lot Area', 'Year Built', 'Garage Cars', 'Neighborhood', 'Overall Cond', 'Overall Qual', 'Bsmt Qual', 'Heating', 'Central Air', 'bathrooms']

Target statistics:
count     2930.00
mean    180796.06
std      79886.69
min      12789.00
25%     129500.00
50%     160000.00
75%     213500.00
max     755000.00
Name: SalePrice, dtype: float64


What we conclude:

    Features: 2930 rows × 12 columns

    Target: SalePrice ranges from $12,789 to $755,000

    Mean ($180,796) > Median ($160,000) confirms right skew

CELL 4: THREE-WAY SPLIT (CRITICAL - NO LEAKAGE)

Explanation: This is the MOST IMPORTANT cell for preventing data leakage. We split FIRST before any preprocessing. Train (70%) learns patterns. Validation (15%) helps tune hyperparameters. Test (15%) is used ONLY ONCE at the end.